In [ ]:
import os
from tqdm import tqdm
import requests
import json 

# Descargar el listado de datasets
list_url = "https://web-api.fiscalia.gob.bo/v1/publico/datasets/list"
list_payload = {
    "size": 100,  # Ajusta si hay más de 100 datasets
    "page": 1,
    "orderBy": "id",
    "orderDirection": "desc",
    "where": {},
    "select": {"id": True}
}
list_headers = {"Content-Type": "application/json"}
resp = requests.post(list_url, json=list_payload, headers=list_headers)
resp.raise_for_status()
datasets = resp.json()['response']['data']

print(f"Se encontraron {len(datasets)} datasets.")

# Crear carpeta de salida si no existe
os.makedirs('../data/datasets', exist_ok=True)

dataset_urls = [d['url'] for d in datasets]
print(dataset_urls)

Se encontraron 10 datasets.
['delitos-por-robo-2025', 'violencia-domestica-2025', 'casos-registrados-2025', 'casos-cerrados-2025', 'personas-en-rebeldia', 'delitos-en-ambientes-digitales', 'delitos-por-robo-2024', 'violencia-domestica-2024', 'casos-registrados-2024', 'casos-cerrados-2024']


In [ ]:
# Descargar y generar datapackage.json para cada dataset del listado
for dataset_url in tqdm(dataset_urls, desc="Procesando datasets"):
    meta_url = f"https://web-api.fiscalia.gob.bo/v1/publico/datasets/{dataset_url}"
    try:
        meta_resp = requests.get(meta_url)
        meta_resp.raise_for_status()
        meta = meta_resp.json()['response']['data']
    except requests.HTTPError as e:
        print(f"Saltando {dataset_url} por error: {e}")
        continue

    # Crear carpeta específica para el dataset
    dataset_folder = f"../data/{meta['nombre'].lower().replace(' ', '-')}"
    os.makedirs(dataset_folder, exist_ok=True)

    # Construir descriptor datapackage
    descriptor = {
        "name": meta['nombre'].lower().replace(' ', '-'),
        "title": meta['nombre'],
        "description": meta['descripcion'],
        "licenses": [{
            "name": meta.get('licencia', 'N/A'),
            "title": meta.get('licencia', 'N/A'),
            "path": ""
        }],
        "sources": [{
            "title": meta.get('fuente', 'N/A'),
            "path": "https://portales.mp.gob.bo/datasets/" + meta.get('url', '')
        }],
        "resources": [],
        "keywords": meta.get('tags', [])
    }

    # Descargar archivos de datos
    for archivo in meta['archivos']:
        file_path = os.path.join(dataset_folder, archivo['fileName'])
        file_url = f"https://files.mp.gob.bo/v1/file/download/{archivo['url']}"
        try:
            if not os.path.exists(file_path):
                r = requests.get(file_url, stream=True)
                if r.status_code == 200:
                    with open(file_path, 'wb') as f:
                        for chunk in r.iter_content(chunk_size=8192):
                            f.write(chunk)
                else:
                    print(f"  Archivo no descargable: {archivo['fileName']} (status {r.status_code})")
                    continue
        except Exception as e:
            print(f"  Error descargando {archivo['fileName']}: {e}")
            continue
        descriptor['resources'].append({
            "path": archivo['fileName'],
            "name": archivo['fileName'],
            "format": archivo['extension'],
            "mediatype": f"text/{archivo['extension']}" if archivo['extension'] in ['csv', 'json', 'xml'] else '',
            "bytes": int(archivo['size']),
            "schema": {
                "fields": [
                    {
                        "name": col['nombreColumn'],
                        "type": col['tipoDato'],
                        "description": col['descripcion']
                    } for col in meta['datasetColumns']
                ]
            }
        })

    # Guardar datapackage.json en la carpeta del dataset
    dp_path = os.path.join(dataset_folder, 'datapackage.json')
    with open(dp_path, 'w', encoding='utf-8') as f:
        json.dump(descriptor, f, ensure_ascii=False, indent=2)

    print(f"Procesado: {meta['nombre']}")

Procesando datasets:   0%|          | 0/10 [00:00<?, ?it/s]

Procesando datasets:  10%|█         | 1/10 [00:00<00:02,  3.57it/s]

Procesado: DELITOS POR ROBO 2025


Procesando datasets:  20%|██        | 2/10 [00:00<00:03,  2.52it/s]

Procesado: VIOLENCIA DOMESTICA 2025


...

Procesado: DELITOS EN AMBIENTES DIGITALES


Procesando datasets:  90%|█████████ | 9/10 [00:02<00:00,  5.67it/s]

Saltando delitos-por-robo-2024 por error: 422 Client Error: Unprocessable Entity for url: https://web-api.fiscalia.gob.bo/v1/publico/datasets/delitos-por-robo-2024
Saltando violencia-domestica-2024 por error: 422 Client Error: Unprocessable Entity for url: https://web-api.fiscalia.gob.bo/v1/publico/datasets/violencia-domestica-2024
Saltando casos-registrados-2024 por error: 422 Client Error: Unprocessable Entity for url: https://web-api.fiscalia.gob.bo/v1/publico/datasets/casos-registrados-2024


Procesando datasets: 100%|██████████| 10/10 [00:02<00:00,  4.44it/s]

Saltando casos-cerrados-2024 por error: 422 Client Error: Unprocessable Entity for url: https://web-api.fiscalia.gob.bo/v1/publico/datasets/casos-cerrados-2024
